# The Bayon face, one face at a time

An homage to home. Head-on, the cube tessellation shows a smiling face of the Bayon, Angkor's temple of
faces. Each of its 25 faces is painted separately, from its own prompt, while following the lines of
the face it covers. Tilted toward any axis, the face breaks apart into that axis's paintings.

**How it works**

1. **A photo** of a Bayon face sets the composition. You place the model on it.
2. **The guide:** a line drawing of the whole photo, made once and saved. Each face cuts out the part it
   covers and unshears it into an upright square, so the lines run on across face borders. It's a plain
   image: clean it up in any paint program, or replace it with your own drawing.
3. **Painting:** Stable Diffusion XL with a canny ControlNet paints a face from its prompt, following its
   piece of the guide. Every painting is saved as a numbered **candidate**.
4. **Keeping:** when you like a candidate, keep it. Kept faces are never repainted, so you can try new
   prompts on the rest as often as you like.

Everything lives in one folder on your Google Drive, so it survives Colab disconnects:

```
MyDrive/folded-illusions/bayon/
  bayon.jpg                          the photo
  guide.png                          the line drawing; edit freely
  candidates/face_07/003.png .json   every painting of face 7, with the prompt, seed and settings
  kept/face_07.png .json             the one you kept
```

The loop: edit `PROMPTS`, paint a few faces, look, keep the good ones, repeat. The board at the end shows
the whole model with what you've kept so far.

**Like cubism.ipynb, this is virtual:** ideal square faces, no visibility map from real paper yet.

## Setup

Use a **GPU runtime** for painting (Runtime → Change runtime type → T4). Placing the photo and the board
run on CPU too. The first cell asks to connect Google Drive.

In [ ]:
import sys, os, subprocess
from pathlib import Path

BRANCH = "main"                  # the repo branch with bayon.py
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("folded-illusions").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/rath-hy/folded-illusions.git"], check=True)
    sys.path.insert(0, "folded-illusions/cubism")
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/folded-illusions/bayon")
else:
    sys.path.insert(0, ".")
    WORK = Path("bayon-work")         # next to the notebook, when run locally

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import bayon as b

store = b.Store(WORK)
print("working folder:", WORK)

In [ ]:
%%capture
# Painting and the lineart guide need these; skip on CPU-only runs.
!pip install -q -U diffusers transformers accelerate controlnet_aux

## The photo

Put the photo in the working folder as `bayon.jpg` (any image type works; change `PHOTO`). If it isn't
there, this cell asks you to upload it. Record where it came from in the README if we ever publish
results made from it.

In [ ]:
PHOTO = WORK / "bayon.jpg"

if not PHOTO.exists() and IN_COLAB:
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    Image.open(name).convert("RGB").save(PHOTO, quality=95)
photo = np.asarray(Image.open(PHOTO).convert("RGB"))
print(photo.shape)

## Place the model

`FIT` puts the model on the photo, head-on: `x` and `y` are its center as fractions of the photo's width
and height, and `width` is how much of the photo's width it spans. Left: the faces on the photo. Right:
what the model shows head-on if each face is printed with its piece.

Faces are numbered 1 to 25 in reading order, top to bottom and left to right. Any part of a face that
falls outside the photo is black, so keep the outlines inside it.

In [ ]:
FIT = dict(x=0.5, y=0.5, width=0.88)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].imshow(b.outlined_photo(photo, FIT)); axes[0].set_title("the faces on the photo")
axes[1].imshow(b.numbered(b.head_on_photo(photo, FIT, 768), 768)); axes[1].set_title("the model head-on")
for ax in axes:
    ax.axis("off")
plt.show()

## The guide

The line drawing every face follows. It's made once and saved as `guide.png`; after that it's loaded
from the file, so your edits stick. To make it again from the photo, set `REMAKE = True`.

- **`method="lineart"`** (default): a learned line detector run on a smoothed copy of the photo. It draws
  the contours of the face, eyes, nose, lips and crown, and much less of the stone's texture than canny.
  - `smooth`: rounds of edge-preserving blur first. More drops more texture.
  - `resolution`: the size the detector works at. Lower gives simpler, bolder lines.
  - `threshold`: 0 to 255. Higher keeps only the stronger lines.
- **`method="canny"`**: plain canny, with `blur`, `low` and `high`. On weathered stone it mostly finds
  texture.
- **`min_area`**: drops bits of line smaller than this many pixels.

**To edit it:** download `guide.png` from the Drive folder, paint over it (white lines on black, or black
on white: either works), save it back under the same name, and rerun this cell. Painting out stray block
joints and strengthening the eyes, nose and smile helps the most.

`LINE_WIDTH` is how thick the lines are on a 1024-pixel face, as the painter sees them.

In [ ]:
GUIDE = dict(method="lineart", smooth=4, resolution=384, threshold=80, min_area=40, blur=2.5, low=40, high=110)
REMAKE = False
b.LINE_WIDTH = 6

if store.guide_path.exists() and not REMAKE:
    guide = b.load_guide(store.guide_path, photo.shape)
    print("loaded", store.guide_path)
else:
    guide = b.make_guide(photo, GUIDE)
    Image.fromarray(guide).save(store.guide_path)
    print("made and saved", store.guide_path)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].imshow(b.outlined_photo(guide, FIT)); axes[0].set_title("the guide")
tex = [b.to_texture(n, b.face_edges(guide, n, FIT, 384), 256) for n in b._in_faces_order()]
axes[1].imshow(b.numbered(b.render(tex, size=768, bg=0.0), 768)); axes[1].set_title("cut into faces, head-on")
for ax in axes:
    ax.axis("off")
plt.show()

**One face up close.** Each face is painted as an upright square: as you'd see it with the model tilted
straight toward that face's axis, turned so its top is as close to up as it can be. Head-on it's
sheared, which is why the photo's piece looks slanted here. Change `LOOK` to see another face.

In [ ]:
LOOK = 17

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(b.face_photo(photo, LOOK, FIT, 512)); axes[0].set_title(f"face {LOOK}: the photo")
axes[1].imshow(b.face_edges(guide, LOOK, FIT, 512), cmap="gray"); axes[1].set_title(f"face {LOOK}: its guide")
for ax in axes:
    ax.axis("off")
plt.show()

## Prompts

One prompt per face. `STYLE` is added to every prompt, which keeps 25 separate paintings looking like
one piece; `NEGATIVE` steers away from things on every face. The starters below are placeholders: make
them yours.

Changing a prompt never touches a face you've kept. Every candidate records the prompt that made it.

In [ ]:
STYLE = ("a synthetic cubist painting, bold flat colors, sharp geometric shapes with hard straight edges, "
         "saffron, lotus pink, jade green, temple gold, deep red and indigo")
NEGATIVE = "blurry, soft edges, gradients, photo, realistic, muddy colors, text, letters, watermark, signature"

PROMPTS = {
    1: "the five lotus towers of Angkor Wat at sunrise",
    2: "an apsara dancer carved in sandstone",
    3: "a pink lotus in bloom",
    4: "a seven-headed naga serpent",
    5: "sugar palm trees against the sky",
    6: "the Mekong river at dusk with long fishing boats",
    7: "a monk in saffron robes",
    8: "a red and white checked krama scarf",
    9: "a golden seated Buddha",
    10: "rice paddies with a water buffalo",
    11: "a Khmer temple roof with curling finials",
    12: "houses on stilts in a floating village",
    13: "a tuk-tuk on a red dirt road",
    14: "Garuda with spread wings",
    15: "fishermen casting nets on the Tonle Sap lake",
    16: "jasmine flower offerings",
    17: "the churning of the ocean of milk, a temple bas-relief",
    18: "a Khmer silk weaving loom",
    19: "green pepper vines in Kampot",
    20: "the water festival boat races",
    21: "a lotus pond with lily pads",
    22: "a hamsa, the sacred goose",
    23: "banyan tree roots over temple stones",
    24: "candles and incense at a shrine",
    25: "a stone lion guarding a temple stair",
}
assert sorted(PROMPTS) == b.NUMBERS, "one prompt per face, 1 to 25"

labels = {n: ("KEPT" if n in store.kept() else "") for n in b.NUMBERS}
plt.figure(figsize=(8, 8))
plt.imshow(b.numbered(b.render(b.board(store, guide, FIT), size=768, light=b.LIT), 768, labels=labels))
plt.axis("off"); plt.title("faces so far"); plt.show()
for n in b.NUMBERS:
    print(f"{n:>2} {'kept' if n in store.kept() else '    '}  {PROMPTS[n]}")

## Paint

`PAINT` is which faces to paint this run: a list of face numbers, `"unkept"` for every face you haven't
kept yet, or `"all"`. A face you list by number is painted even if it's kept, which only adds
candidates; your kept choice stays until you change it.

Each face gets `PER_FACE` new candidates with fresh random seeds (or set `SEEDS` to repeat exact ones).
On a T4, each painting takes about a minute, so a full `"unkept"` round of 25 faces at 2 each is about
50 minutes. Start with a few.

- `control_scale`: how strictly the painting follows the guide. Higher keeps the face's lines; lower
  gives the prompt more room.
- `size`: 1024 is SDXL's native size. 768 is about twice as fast, for quick looks.

In [ ]:
PAINT = [13, 17, 18]
PER_FACE = 2
SEEDS = None                      # or a list, e.g. [0, 1], to repeat exact seeds
SETTINGS = dict(control_scale=0.6, guidance=7.0, steps=30, size=1024)

if "painter" not in globals():
    painter = b.load_painter()

rng = np.random.default_rng()
for n in b.todo(store, PAINT):
    seeds = SEEDS if SEEDS is not None else [int(s) for s in rng.integers(0, 2**31, PER_FACE)]
    made = b.paint(painter, store, guide, n, PROMPTS[n], seeds, STYLE, NEGATIVE, FIT, SETTINGS)
    fig, axes = plt.subplots(1, len(made) + 1, figsize=(4 * (len(made) + 1), 4.3))
    axes[0].imshow(b.face_edges(guide, n, FIT, 512), cmap="gray"); axes[0].set_title(f"face {n}: guide")
    for ax, c in zip(axes[1:], made):
        pic, info = store.candidate(n, c)
        ax.imshow(pic); ax.set_title(f"face {n}, candidate {c}")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(PROMPTS[n], y=1.02)
    plt.show()

## Keep

`KEEP` maps a face to the candidate you want: `{13: 2}` keeps face 13's candidate 2. Keeping again
replaces your choice. `UNKEEP` sends faces back to unpainted (their candidates stay).

To look back at every candidate of a face, set `BROWSE`. The kept one is outlined in red.

In [ ]:
KEEP = {}                         # e.g. {13: 2, 17: 1}
UNKEEP = []

for n, c in KEEP.items():
    info = store.keep(n, c)
    print(f"kept face {n}: candidate {c}, seed {info['seed']}, \"{info['prompt']}\"")
for n in UNKEEP:
    store.unkeep(n)
    print(f"face {n} is unpainted again")
print(f"{len(store.kept())} of 25 faces kept")

In [ ]:
BROWSE = 13

sheet = b.contact_sheet(store, BROWSE)
if sheet is None:
    print(f"face {BROWSE} has no candidates yet")
else:
    plt.figure(figsize=(14, 14 * sheet.height / sheet.width)); plt.imshow(sheet); plt.axis("off")
    plt.title(f"face {BROWSE}: every candidate"); plt.show()

## The board

The whole model with what you've kept. Faces not yet kept show their guide lines in gray. Head-on (lit
as paper would be) is the artwork; each tilt is a jumble of one axis's paintings. Saved to the working
folder as `board.gif`.

In [ ]:
tex = b.board(store, guide, FIT)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(b.render(tex, size=512, light=b.LIT)); axes[0].set_title("head-on")
for k in range(3):
    axes[k + 1].imshow(b.render(tex, t=1.0, axis=k, size=384)); axes[k + 1].set_title(f"tilted to {b.AXES[k]}")
for ax in axes:
    ax.axis("off")
plt.show()

plt.figure(figsize=(9, 9)); plt.imshow(b.render(tex, size=900, light=b.LIT)); plt.axis("off"); plt.show()

from IPython.display import Image as ShowImage
b.save_gif(tex, WORK / "board.gif")
ShowImage(filename=str(WORK / "board.gif"))

## What this leaves out

- **The real visibility map.** Like the guitar, this paints ideal cube faces. Printing needs the map from
  the folded sheet to each face, and the model's rim has flat triangles we haven't painted.
- **Lighting.** Real light makes the three axes' faces different brightnesses (`LIT` imitates it), so a
  face's colors shift by axis when folded. We could pre-brighten the darker axes before printing.
- **Seams.** Neighboring faces are painted separately, so their colors don't continue across borders,
  only the guide's lines do. That suits cubism, but a shared `STYLE` palette matters.